In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("cleaned_welface.csv")
df

,content,label
0,law enforcement on high alert following threat...,1
1,unbelievable obamas attorney general says most...,1
2,bobby jindal raised hindu uses story of christ...,0
3,satan 2 russia unvelis an image of its terrify...,1
4,about time christian group sues amazon and spl...,1
...,...,...
63066,wikileaks email shows clinton foundation funds...,1
63067,russians steal research on trump in hack of us...,0
63068,watch giuliani demands that democrats apologiz...,1
63069,migrants refuse to leave train at refugee camp...,0


In [2]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

Train: (50456, 2)
Val:   (6307, 2)
Test:  (6308, 2)


In [3]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [4]:
from torch.utils.data import Dataset
import torch

class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            max_length=self.max_length,
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = FakeNewsDataset(
    texts=train_df['content'].tolist(),
    labels=train_df['label'].tolist(),
    tokenizer=tokenizer
)
val_dataset = FakeNewsDataset(
    texts=val_df['content'].tolist(),
    labels=val_df['label'].tolist(),
    tokenizer=tokenizer
)
test_dataset = FakeNewsDataset(
    texts=test_df['content'].tolist(),
    labels=test_df['label'].tolist(),
    tokenizer=tokenizer
)

print("Train samples:", len(train_dataset))
print("Val samples:  ", len(val_dataset))

D:\fake-news-classifier\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train samples: 50456
Val samples:   6307


In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Verify by grabbing one batch
batch = next(iter(train_loader))
print("Input IDs shape:     ", batch['input_ids'].shape)
print("Attention mask shape:", batch['attention_mask'].shape)
print("Labels shape:        ", batch['label'].shape)
print("Sample labels:       ", batch['label'][:5])

Input IDs shape:      torch.Size([16, 256])
Attention mask shape: torch.Size([16, 256])
Labels shape:         torch.Size([16])
Sample labels:        tensor([0, 1, 1, 1, 0])
